# Training-Free Adaptive NeSy-Gen with MedGemma

This notebook replaces Vision–T5 drafting with official MedGemma inference and uses frozen MedSigLIP visual retrieval. The downstream method remains visual RAG + adaptive PrimeKG/LTN verification + Consistency Gate + faithful claim traces.

**Claim boundary:** MedGemma and MedSigLIP pretraining includes MIMIC-CXR. MIMIC experiments must therefore be described as *no task-specific fine-tuning*, not strict unseen-data zero-shot.

## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2: Experiment Configuration

In [ ]:
from pathlib import Path

RUN_DATASET = 'iuxray_official'  # options: iuxray_official, mimic_aug
DRAFT_MODE = 'few_shot'          # options: zero_shot, few_shot
SMOKE_RUN = True                 # verify with 25 cases, then set False
MAX_TEST_EXAMPLES = 25 if SMOKE_RUN else None

MEDGEMMA_MODEL = 'google/medgemma-4b-it'
MEDSIGLIP_MODEL = 'google/medsiglip-448'
RETRIEVAL_TOP_K = 5
FEW_SHOT_K = 3
RETRIEVAL_BATCH_SIZE = 16
MAX_RETRIEVAL_INDEX_EXAMPLES = 5000 if SMOKE_RUN else None
MAX_NEW_TOKENS = 180

SOURCE_RUN_NAME = 'aaai_vision_t5_base_convnext_primekg'
RUN_NAME = f'aaai_adaptive_medgemma_{DRAFT_MODE}'
REPO_DIR = Path('/content/Nesy-GenRevision')
AAAI_ROOT = Path('/content/drive/MyDrive/aaai_2026_experiments')
SOURCE_OUTPUT_DIR = AAAI_ROOT / RUN_DATASET / SOURCE_RUN_NAME
OUTPUT_DIR = AAAI_ROOT / RUN_DATASET / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_NAME = 'iuxray_official_manifest.jsonl' if RUN_DATASET == 'iuxray_official' else 'mimic_aug_manifest.jsonl'
MANIFEST = SOURCE_OUTPUT_DIR / MANIFEST_NAME
RAD_PRIMEKG_DIR = Path(f'/content/drive/MyDrive/primekg_radiology_cache_{RUN_DATASET}')
MEDSIGLIP_CACHE = Path(f'/content/drive/MyDrive/medsiglip_cache_{RUN_DATASET}/train_index.npz')

FAST_ACCEPT_THRESHOLD = 0.85
MIN_SUPPORTING_REPORTS = 2
CLAIM_REVISE_THRESHOLD = 0.50
REVISION_POLICY = 'evidence_replace'

print('Drafting:', MEDGEMMA_MODEL, DRAFT_MODE)
print('Retrieval:', MEDSIGLIP_MODEL)
print('Manifest:', MANIFEST)
print('Output:', OUTPUT_DIR)

## Cell 3: Hugging Face Access

Before running, accept the terms on the official `google/medgemma-4b-it` and `google/medsiglip-448` Hugging Face pages. Add an `HF_TOKEN` secret in Colab.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Add HF_TOKEN under Colab Secrets first.'
login(token=HF_TOKEN)
print('Hugging Face authentication ready.')

## Cell 4: Clone/Pull Repository and Install

In [ ]:
import subprocess, sys

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/FaezehMillerAI/Nesy-GenRevision.git', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[torch,eval,colab]'], check=True)
print('Repository ready:', REPO_DIR)

## Cell 5: Verify GPU and Existing Data Assets

In [ ]:
import torch

required = [MANIFEST, RAD_PRIMEKG_DIR / 'kg.csv', RAD_PRIMEKG_DIR / 'nodes.csv']
for path in required: print(path, '->', path.exists())
assert all(path.exists() for path in required), 'Correct missing paths in Cell 2.'
assert torch.cuda.is_available(), 'Select a GPU runtime.'
print('GPU:', torch.cuda.get_device_name(0))
print('MedSigLIP cache already exists:', MEDSIGLIP_CACHE.exists())

## Cell 6: Generate and Adaptively Verify Reports

The first run builds the frozen MedSigLIP training index. Later runs reuse it from Drive. No gradient training occurs.

In [ ]:
import os, subprocess, sys

PREDICTIONS_CSV = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_predictions.csv'
CANDIDATES_CSV = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_retrieval_audit.csv'
CLAIM_TRACE_JSONL = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_claim_traces.jsonl'
CLAIM_AUDIT_CSV = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_claim_audit.csv'
RUN_LOG = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_run.log'

cmd = [sys.executable, '-u', 'scripts/generate_medgemma_adaptive_reports.py',
    '--manifest', str(MANIFEST), '--primekg-dir', str(RAD_PRIMEKG_DIR),
    '--output-csv', str(PREDICTIONS_CSV), '--candidates-csv', str(CANDIDATES_CSV),
    '--claim-trace-jsonl', str(CLAIM_TRACE_JSONL), '--claim-audit-csv', str(CLAIM_AUDIT_CSV),
    '--draft-mode', DRAFT_MODE, '--medgemma-model', MEDGEMMA_MODEL,
    '--medsiglip-model', MEDSIGLIP_MODEL, '--retrieval-top-k', str(RETRIEVAL_TOP_K),
    '--few-shot-k', str(FEW_SHOT_K), '--retrieval-batch-size', str(RETRIEVAL_BATCH_SIZE),
    '--retrieval-cache', str(MEDSIGLIP_CACHE), '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--fast-accept-threshold', str(FAST_ACCEPT_THRESHOLD),
    '--min-supporting-reports', str(MIN_SUPPORTING_REPORTS),
    '--claim-revise-threshold', str(CLAIM_REVISE_THRESHOLD),
    '--revision-policy', REVISION_POLICY]
if MAX_TEST_EXAMPLES: cmd += ['--limit', str(MAX_TEST_EXAMPLES)]
if MAX_RETRIEVAL_INDEX_EXAMPLES: cmd += ['--max-retrieval-index-examples', str(MAX_RETRIEVAL_INDEX_EXAMPLES)]

env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
with RUN_LOG.open('w', encoding='utf-8') as log_file:
    process = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=0, env=env)
    while True:
        chunk = process.stdout.read(1)
        if chunk == '' and process.poll() is not None: break
        if chunk: print(chunk, end='', flush=True); log_file.write(chunk); log_file.flush()
    return_code = process.wait()
if return_code != 0: raise RuntimeError(f'Run failed. Inspect {RUN_LOG}')
print('Predictions:', PREDICTIONS_CSV)

## Cell 7: Explainability and Routing Metrics

In [ ]:
import json, subprocess, sys

EXPLAIN_JSON = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_explainability.json'
EXPLAIN_CSV = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_explainability_claims.csv'
subprocess.run([sys.executable, 'scripts/evaluate_adaptive_explanations.py', '--trace-jsonl', str(CLAIM_TRACE_JSONL),
    '--output-json', str(EXPLAIN_JSON), '--output-csv', str(EXPLAIN_CSV)], cwd=REPO_DIR, check=True)
json.loads(EXPLAIN_JSON.read_text())

## Cell 8: Generation and Leakage Evaluation

In [ ]:
import json, subprocess, sys

METRICS_JSON = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_metrics.json'
LEAKAGE_JSON = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_leakage.json'
subprocess.run([sys.executable, 'scripts/evaluate_generation.py', '--manifest', str(MANIFEST),
    '--predictions-csv', str(PREDICTIONS_CSV), '--output-json', str(METRICS_JSON),
    '--nodes-csv', str(RAD_PRIMEKG_DIR / 'nodes.csv'),
    '--output-factuality-csv', str(OUTPUT_DIR / 'entity_factuality.csv'),
    '--output-chexpert-csv', str(OUTPUT_DIR / 'chexpert_quick.csv'),
    '--output-radgraph-csv', str(OUTPUT_DIR / 'radgraph_quick.csv')], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, 'scripts/audit_prediction_leakage.py', '--manifest', str(MANIFEST),
    '--predictions-csv', str(PREDICTIONS_CSV), '--output-json', str(LEAKAGE_JSON),
    '--high-overlap-threshold', '0.95'], cwd=REPO_DIR, check=True)
print(json.dumps(json.loads(METRICS_JSON.read_text())['lexical_metrics'], indent=2))
print(LEAKAGE_JSON.read_text())

## Cell 9: Readable Faithful Explanation Report

In [ ]:
import subprocess, sys
from IPython.display import HTML, display

REPORT_HTML = OUTPUT_DIR / f'{RUN_DATASET}_{DRAFT_MODE}_explanations.html'
subprocess.run([sys.executable, 'scripts/build_adaptive_explanation_report.py',
    '--trace-jsonl', str(CLAIM_TRACE_JSONL), '--output-html', str(REPORT_HTML), '--limit', '20'],
    cwd=REPO_DIR, check=True)
display(HTML(REPORT_HTML.read_text()))

## Cell 10: Prepare Official Clinical Metric Inputs

In [ ]:
import subprocess, sys

OFFICIAL_INPUT_DIR = OUTPUT_DIR / 'official_metric_inputs'
subprocess.run([sys.executable, 'scripts/evaluate_generation.py', '--manifest', str(MANIFEST),
    '--predictions-csv', str(PREDICTIONS_CSV), '--output-json', str(OUTPUT_DIR / 'official_input_prep.json'),
    '--prepare-official-inputs-dir', str(OFFICIAL_INPUT_DIR),
    '--nodes-csv', str(RAD_PRIMEKG_DIR / 'nodes.csv')], cwd=REPO_DIR, check=True)
print('Official CheXbert/RadGraph inputs:', OFFICIAL_INPUT_DIR)